In [1]:
!pip install pycryptodome

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 27.5 MB/s eta 0:00:00


In [2]:
import base64
import binascii
from Crypto.Util.number import getPrime

def egcd(a, b):
    if a == 0: return (b, 0, 1)
    g, y, x = egcd(b % a, a)
    return (g, x - (b // a) * y, y)

def modinv(a, m):
    g, x, y = egcd(a, m)
    if g != 1: raise Exception('Không tìm thấy nghịch đảo modulo')
    return x % m

def generate_rsa_keys(p, q, e):
    n = p * q
    phi = (p - 1) * (q - 1)
    d = modinv(e, phi)
    return (e, n), (d, n) # PU = (e, n), PR = (d, n)

print("--- KHÓA NGẪU NHIÊN LỚN ---")
p_rand = getPrime(1024)
q_rand = getPrime(1024)
e_rand = 65537
PU_rand, PR_rand = generate_rsa_keys(p_rand, q_rand, e_rand)
print(f"p ngẫu nhiên: {p_rand}\n")
print(f"q ngẫu nhiên: {q_rand}\n")
print(f"n ngẫu nhiên: {PU_rand[1]}\n")

--- KHÓA NGẪU NHIÊN LỚN ---
p ngẫu nhiên: 114719423249070250707312608163402583565482491665225371537282837593412422495594836937580734479071238242113660677086435590068810586205298029730549715888823200877436116051437102804526239067586855786657430335873248472814472908508310541815708888630633427603469617366998535265259499082263041150120440343964432627097

q ngẫu nhiên: 176682566835741127704391769764269958640932116423129261672386214780171691601387759645795001185357289355380372503955975186623156248549014981393340382831539925366558160332177352266423880645435408358580717217948678849132541684848257774702877789349848448797436420755001898228321712920387750020810457978287732471727

n ngẫu nhiên: 20268922165561529165006697171513432406046287084431247402698183955331676327958413128537436187861152376219350003689221090215941725656783057257205476151768586407392753498077984478896513167891631458475021656257085058852290930072172635819972063202316596119686538951934437192888983490400788980627555696612660874304

In [3]:
print("\n--- YÊU CẦU 1: XÁC ĐỊNH KHÓA PU VÀ PR ---")

p1, q1, e1 = 11, 17, 7
PU1, PR1 = generate_rsa_keys(p1, q1, e1)
print(f"Trường hợp 1:\n PU1 = {PU1}\n PR1 = {PR1}\n")

p2 = 20079993872842322116151219
q2 = 676717145751736242170789
e2 = 17
PU2, PR2 = generate_rsa_keys(p2, q2, e2)
print(f"Trường hợp 2:\n PU2 = (e2: {PU2[0]},\n n2: {PU2[1]})\n PR2 = (d2: {PR2[0]},\n n2: {PR2[1]})\n")

p3 = int("F7E75FDC469067FFDC4E847C51F452DF", 16)
q3 = int("E85CED54AF57E53E092113E62F436F4F", 16)
e3 = int("0D88C3", 16)
PU3, PR3 = generate_rsa_keys(p3, q3, e3)
print(f"Trường hợp 3:\n PU3 = (e3: {hex(PU3[0])},\n n3: {hex(PU3[1])})\n PR3 = (d3: {hex(PR3[0])},\n n3: {hex(PR3[1])})\n")


--- YÊU CẦU 1: XÁC ĐỊNH KHÓA PU VÀ PR ---
Trường hợp 1:
 PU1 = (7, 187)
 PR1 = (23, 187)

Trường hợp 2:
 PU2 = (e2: 17,
 n2: 13588476140342208394395166469647627226674348541791)
 PR2 = (d2: 7993221259024828467291262184080358019185876599873,
 n2: 13588476140342208394395166469647627226674348541791)

Trường hợp 3:
 PU3 = (e3: 0xd88c3,
 n3: 0xe103abd94892e3e74afd724bf28e78366d9676bccc70118bd0aa1968dbb143d1)
 PR3 = (d3: 0x3587a24598e5f2a21db007d89d18cc50aba5075ba19a33890fe7c28a9b496aeb,
 n3: 0xe103abd94892e3e74afd724bf28e78366d9676bccc70118bd0aa1968dbb143d1)



In [4]:
print("\n--- YÊU CẦU 2: BẢO MẬT VÀ XÁC THỰC VỚI M=5 (Dùng Khóa 1) ---")
M = 5
e1, n1 = PU1
d1, _ = PR1

C_conf = pow(M, e1, n1)
M_dec_conf = pow(C_conf, d1, n1)
print(f"Bảo mật - Bản mã C: {C_conf}")
print(f"Bảo mật - Giải mã lại M: {M_dec_conf}")

C_auth = pow(M, d1, n1)
M_dec_auth = pow(C_auth, e1, n1)
print(f"Xác thực - Bản mã C: {C_auth}")
print(f"Xác thực - Giải mã lại M: {M_dec_auth}")


--- YÊU CẦU 2: BẢO MẬT VÀ XÁC THỰC VỚI M=5 (Dùng Khóa 1) ---
Bảo mật - Bản mã C: 146
Bảo mật - Giải mã lại M: 5
Xác thực - Bản mã C: 180
Xác thực - Giải mã lại M: 5


In [9]:
print("\n--- YÊU CẦU 3: MÃ HÓA CHUỖI VÀ XUẤT BASE64 ---")
message = "The University of Information Technology"
M_text = int.from_bytes(message.encode('utf-8'), 'big')


e3, n3 = PU3
if M_text >= n3:
    print("Cảnh báo: Thông điệp lớn hơn module n. Trong thực tế cần chia khối (block) hoặc dùng PKCS#1 v1.5 padding.")

C_text = pow(M_text, e3, n3)

C_bytes = C_text.to_bytes((C_text.bit_length() + 7) // 8, 'big')
C_base64 = base64.b64encode(C_bytes).decode('utf-8')

print(f"Bản mã Base64: {C_base64}")


--- YÊU CẦU 3: MÃ HÓA CHUỖI VÀ XUẤT BASE64 ---
Cảnh báo: Thông điệp lớn hơn module n. Trong thực tế cần chia khối (block) hoặc dùng PKCS#1 v1.5 padding.
Bản mã Base64: zGSz6Z2vJ30XKC/GZfT8r+Hn5r9IhIW2+PtRG0A7nxc=


In [52]:
import base64, math

# -----------------------------------------------------------------
# 1. XÂY DỰNG KHÓA
# -----------------------------------------------------------------
def ext_gcd(a, b):
    if a == 0: return b, 0, 1
    g, x, y = ext_gcd(b % a, a)
    return g, y - (b // a) * x, x

def modinv(e, phi):
    _, x, _ = ext_gcd(e % phi, phi)
    return x % phi

def make_key(p, q, e):
    n = p * q
    d = modinv(e, (p-1)*(q-1))
    return dict(n=n, e=e, d=d, bits=n.bit_length())

K1 = make_key(11, 17, 7)
K2 = make_key(20079993872842322116151219,
               676717145751736242170789, 17)
K3 = make_key(int("F7E75FDC469067FFDC4E847C51F452DF",16),
               int("E85CED54AF57E53E092113E62F436F4F",16),
               int("0D88C3",16))

print("=== THÔNG TIN KHÓA ===")
for name, k in [("K1",K1),("K2",K2),("K3",K3)]:
    print(f"{name}: n={k['bits']} bit, e={k['e']}")
    print(f"     d={k['d']}")

# -----------------------------------------------------------------
# 2. HÀM TIỆN ÍCH
# -----------------------------------------------------------------
def rsa_dec_int(c, key):
    """Giải mã RSA: M = C^d mod n"""
    return pow(c, key['d'], key['n'])

def decode_blocks(raw_bytes, key, block_size):
    """
    Chia raw_bytes thành từng block block_size bytes,
    giải mã RSA từng block, ghép lại thành plaintext bytes.
    """
    result = b""
    n_blocks = math.ceil(len(raw_bytes) / block_size)
    for i in range(n_blocks):
        blk = raw_bytes[i*block_size : (i+1)*block_size]
        c   = int.from_bytes(blk, 'big')
        m   = rsa_dec_int(c, key)
        # Đổi số nguyên → bytes, giữ leading bytes nếu cần
        m_len = max(1, math.ceil(m.bit_length()/8))
        result += m.to_bytes(m_len, 'big')
    return result

def pretty(b):
    try:
        s = b.decode('utf-8')
        if all(32 <= ord(c) <= 126 or c in '\n\r\t' for c in s):
            return f'"{s}"'
    except: pass
    try:
        s = b.decode('latin-1')
        printable = sum(1 for c in s if 32<=ord(c)<=126)
        if printable/len(s) > 0.85:
            return f'"{s}"'
    except: pass
    return f""

# -----------------------------------------------------------------
# 3. PHÂN TÍCH KÍCH THƯỚC BẢN MÃ → xác định key và block_size
# -----------------------------------------------------------------
# Quy tắc:
#   block_size = ceil(n.bit_length() / 8)  — kích thước 1 block ciphertext
#   K1: ceil(8/8)  = 1 byte/block  (n=187, 8-bit)
#   K2: ceil(84/8) = 11 byte/block (n~84-bit)
#   K3: ceil(256/8)= 32 byte/block (n~256-bit)
#
#   Bản mã A: base64 → 60 bytes → 60/32 = 1.875 → KHÔNG khớp K3 chia đều
#             60/1=60 (K1), 60/11≈5.45 (K2), 60/30=2 (thử 30-byte block)
#             → Thử K3 với block=30 (bỏ 2 byte leading zero)
#   Bản mã B: hex  → 20 bytes → 20/11=1.8, 20/10=2 → thử K2 block=10
#   Bản mã C: base64→ 48 bytes → 48/32=1.5 (K3 không chia đều)
#             → thử K3 block=24, hoặc K2 block=10 → 48/10=4.8
#             → quan sát hex của C thấy ký tự lặp → khả năng mã hoá từng byte (K1)

print("\n=== PHÂN TÍCH KÍCH THƯỚC ===")
# Bản mã A
ca_b64 = ("raUcesUlOkx/8ZhgodMoo0Uu18sC20yXlQFevSu7W/FDxIy0YRHMyX"\
          "cHdD9PBvIT2aUft5fCQEGomiVVPv4I")
pad = (4 - len(ca_b64)%4) % 4
ca = base64.b64decode(ca_b64 + "="*pad)
print(f"A: {len(ca)} bytes")

# Bản mã B
cb_hex = "C87F570FC4F699CEC24020C6F54221ABAB2CE0C3"
cb = bytes.fromhex(cb_hex)
print(f"B: {len(cb)} bytes")

# Bản mã C (Fixed: full string as in ciphertexts list)
cc_b64 = "Z2BUSkJcg0w4XEpgm0JcMExEQmBlVH6dYEpNTHpMHptMQ7NgTHlgQrNMQ2BKTQ=="
pad2 = (4 - len(cc_b64)%4) % 4 # This will now be 0 since the string is properly padded
cc = base64.b64decode(cc_b64 + "="*pad2)
print(f"C: {len(cc)} bytes")
print(f"   hex: {cc.hex()}")

# Bản mã D (Binary)
cd_bin = "001010000001010011111111101101110010111011001010111011000110011110111111001111110110100011001111001100001001010001010100111101010100110011101110111011110101101100000100"
cd_int = int(cd_bin, 2)
cd = cd_int.to_bytes((cd_int.bit_length() + 7) // 8, 'big')
print(f"D: {len(cd)} bytes")
print(f"   hex: {cd.hex()}")


# -----------------------------------------------------------------
# 4. GIẢI MÃ BẢN MÃ A
# -----------------------------------------------------------------
print("\n" + "="*55)
print("BẢN MÃ A — Base64")
print("="*55)
print(f"Raw ({len(ca)} bytes): {ca.hex()}")
print()

attempts_a = [
    ("K3", K3, 32),
    ("K3", K3, 30),
    ("K2", K2, 21),
    ("K2", K2, 11),
    ("K2", K2, 10),
    ("K1", K1, 1),
]
for kname, key, bs in attempts_a:
    try:
        chunks = []
        ok = True
        # Handle potential partial last block with math.ceil
        n_blocks = math.ceil(len(ca) / bs)
        for i in range(n_blocks):
            blk = ca[i*bs : (i+1)*bs]
            if not blk: continue
            c = int.from_bytes(blk, 'big')
            if c >= key['n']:
                ok = False; break # c must be less than n for valid RSA
            m = rsa_dec_int(c, key)
            mlen = max(1, math.ceil(m.bit_length()/8))
            chunks.append(m.to_bytes(mlen, 'big'))

        if not ok:
            print(f"  {kname} block={bs}B → c >= n (bỏ qua)")
            continue

        pt = b"".join(chunks)
        txt = pretty(pt)
        print(f"  {kname} block={bs}B → {txt}")
    except Exception as ex:
        print(f"  {kname} block={bs}B → lỗi: {ex}")

# -----------------------------------------------------------------
# 5. GIẢI MÃ BẢN MÃ B
# -----------------------------------------------------------------
print("\n" + "="*55)
print("BẢN MÃ B — Hex 20 bytes")
print("="*55)
print(f"Raw ({len(cb)} bytes): {cb.hex()}")
print()

attempts_b = [
    ("K3", K3, 32),   # 20 < 32 → toàn bộ là 1 block nhưng c < n?
    ("K2", K2, 21),
    ("K2", K2, 11),
    ("K2", K2, 10),
    ("K2", K2, 20),   # toàn bộ là 1 block
    ("K1", K1, 1),
]
for kname, key, bs in attempts_b:
    try:
        chunks = []
        ok = True
        n_blocks = math.ceil(len(cb) / bs)
        for i in range(n_blocks):
            blk = cb[i*bs:(i+1)*bs]
            if not blk: continue
            c = int.from_bytes(blk, 'big')
            if c >= key['n']:
                ok = False; break
            m = rsa_dec_int(c, key)
            mlen = max(1, math.ceil(m.bit_length()/8))
            chunks.append(m.to_bytes(mlen,'big'))
        if not ok:
            print(f"  {kname} block={bs}B → c >= n (bỏ qua)")
            continue
        pt = b"".join(chunks)
        txt = pretty(pt)
        print(f"  {kname} block={bs}B → {txt}")
    except Exception as ex:
        print(f"  {kname} block={bs}B → lỗi: {ex}")

# -----------------------------------------------------------------
# 6. GIẢI MÃ BẢN MÃ C
# -----------------------------------------------------------------
print("\n" + "="*55)
print("BẢN MÃ C — Base64")
print("="*55)
print(f"Raw ({len(cc)} bytes): {cc.hex()}")
print()

all_lt_n1 = all(b < K1['n'] for b in cc)
print(f"  Tất cả byte < n1=187? {all_lt_n1}")

attempts_c = [
    ("K1", K1, 1),
    ("K3", K3, 32),
    ("K3", K3, 24),
    ("K2", K2, 21),
    ("K2", K2, 12),
    ("K2", K2, 11),
]
for kname, key, bs in attempts_c:
    try:
        chunks = []
        ok = True
        n_blocks = math.ceil(len(cc) / bs)
        for i in range(n_blocks):
            blk = cc[i*bs:(i+1)*bs]
            if not blk: continue
            c = int.from_bytes(blk, 'big')
            if c >= key['n']:
                ok = False; break
            m = rsa_dec_int(c, key)
            if bs == 1:
                chunks.append(bytes([m % 256]))
            else:
                mlen = max(1, math.ceil(m.bit_length()/8))
                chunks.append(m.to_bytes(mlen,'big'))
        if not ok:
            print(f"  {kname} block={bs}B → c >= n (bỏ qua)")
            continue
        pt = b"".join(chunks)
        txt = pretty(pt)
        print(f"  {kname} block={bs}B → {txt}")
    except Exception as ex:
        print(f"  {kname} block={bs}B → lỗi: {ex}")

# -----------------------------------------------------------------
# 7. GIẢI MÃ BẢN MÃ D
# -----------------------------------------------------------------
print("\n" + "="*55)
print("BẢN MÃ D — Binary")
print("="*55)
print(f"Raw ({len(cd)} bytes): {cd.hex()}")
print()

attempts_d = [
    ("K1", K1, 1),
    ("K3", K3, 32),
    ("K3", K3, 24),
    ("K2", K2, 21),
    ("K2", K2, 12),
    ("K2", K2, 11),
]
for kname, key, bs in attempts_d:
    try:
        chunks = []
        ok = True
        n_blocks = math.ceil(len(cd) / bs)
        for i in range(n_blocks):
            blk = cd[i*bs:(i+1)*bs]
            if not blk: continue
            c = int.from_bytes(blk, 'big')
            if c >= key['n']:
                ok = False; break
            m = rsa_dec_int(c, key)
            if bs == 1:
                chunks.append(bytes([m % 256]))
            else:
                mlen = max(1, math.ceil(m.bit_length()/8))
                chunks.append(m.to_bytes(mlen,'big'))
        if not ok:
            print(f"  {kname} block={bs}B → c >= n (bỏ qua)")
            continue
        pt = b"".join(chunks)
        txt = pretty(pt)
        print(f"  {kname} block={bs}B → {txt}")
    except Exception as ex:
        print(f"  {kname} block={bs}B → lỗi: {ex}")


# -----------------------------------------------------------------
# 8. TÓM TẮT
# -----------------------------------------------------------------
print("\n" + "="*55)
print("TÓM TẮT — kết quả có nghĩa nhất")
print("="*55)
def best_decode(raw, attempts):
    for kname, key, bs in attempts:
        chunks = []
        ok = True
        n_blocks = math.ceil(len(raw) / bs)
        for i in range(n_blocks):
            blk = raw[i*bs:(i+1)*bs]
            if not blk: continue
            c = int.from_bytes(blk, 'big')
            if c >= key['n']: ok=False; break
            m = rsa_dec_int(c, key)
            if bs == 1:
                chunks.append(bytes([m % 256]))
            else:
                mlen = max(1, math.ceil(m.bit_length()/8))
                chunks.append(m.to_bytes(mlen,'big'))
        if not ok: continue
        pt = b"".join(chunks)
        try:
            s = pt.decode('utf-8')
            if sum(1 for c in s if 32<=ord(c)<=126)/max(len(s),1) > 0.8:
                return kname, bs, s
        except: pass
    return None, None, None

kA, bA, pA = best_decode(ca, attempts_a)
kB, bB, pB = best_decode(cb, attempts_b)
kC, bC, pC = best_decode(cc, attempts_c)
kD, bD, pD = best_decode(cd, attempts_d) # Add Bản mã D here

print(f"A: khóa={kA}, block={bA}B → {repr(pA)}")
print(f"B: khóa={kB}, block={bB}B → {repr(pB)}")
print(f"C: khóa={kC}, block={bC}B → {repr(pC)}")
print(f"D: khóa={kD}, block={bD}B → {repr(pD)}") # Print Bản mã D result

=== THÔNG TIN KHÓA ===
K1: n=8 bit, e=7
     d=23
K2: n=164 bit, e=17
     d=7993221259024828467291262184080358019185876599873
K3: n=256 bit, e=886979
     d=24212225287904763939160097464943268930139828978795606022583874367720623008491

=== PHÂN TÍCH KÍCH THƯỚC ===
A: 63 bytes
B: 20 bytes
C: 46 bytes
   hex: 6760544a425c834c385c4a609b425c304c44426065547e9d604a4d4c7a4c1e9b4c43b3604c796042b34c43604a4d
D: 21 bytes
   hex: 2814ffb72ecaec67bf3f68cf309454f54ceeef5b04

BẢN MÃ A — Base64
Raw (63 bytes): ada51c7ac5253a4c7ff19860a1d328a3452ed7cb02db4c9795015ebd2bbb5bf143c48cb46111ccc97707743f4f06f213d9a51fb797c24041a89a25553efe08

  K3 block=32B → 
  K3 block=30B → 
  K2 block=21B → c >= n (bỏ qua)
  K2 block=11B → 
  K2 block=10B → 
  K1 block=1B → c >= n (bỏ qua)

BẢN MÃ B — Hex 20 bytes
Raw (20 bytes): c87f570fc4f699cec24020c6f54221abab2ce0c3

  K3 block=32B → 
  K2 block=21B → 
  K2 block=11B → 
  K2 block=10B → 
  K2 block=20B → 
  K1 block=1B → c >= n (bỏ qua)

BẢN MÃ C — Base64
Raw (46 by

In [33]:
!pip install cryptography

In [35]:
with open('hash_tbs.txt', 'r') as f:
    hash_tbs = f.read().strip()

EM_int_verified = pow(s_extracted, e_extracted, n_extracted)

EM_hex_verified = hex(EM_int_verified)[2:]

if len(EM_hex_verified) % 2 != 0:
    EM_hex_verified = '0' + EM_hex_verified

print("="*60)
print("KẾT QUẢ XÁC MINH CHỨNG CHỈ BẰNG PYTHON (RSA) - Cập nhật")
print("="*60)
print(f"1. Mã băm tính từ file c0_body.bin: \n   {hash_tbs}")
print(f"\n2. Kết quả RSA (S^e mod n) giải mã từ Chữ ký: \n   ...{EM_hex_verified[-64:]}")

if EM_hex_verified.endswith(hash_tbs):
    print("\n[✓] XÁC MINH THÀNH CÔNG! ")
    print("Mã băm trong chữ ký khớp hoàn toàn với mã băm của nội dung chứng chỉ.")
else:
    print("\n[x] XÁC MINH THẤT BẠI!")
    print("Mã băm không khớp.")


KẾT QUẢ XÁC MINH CHỨNG CHỈ BẰNG PYTHON (RSA) - Cập nhật
1. Mã băm tính từ file c0_body.bin: 
   5675fbaad2975225a4fc43ec1f3031c7408bd43acc51aef85dbab34d762c0a34

2. Kết quả RSA (S^e mod n) giải mã từ Chữ ký: 
   ...5675fbaad2975225a4fc43ec1f3031c7408bd43acc51aef85dbab34d762c0a34

[✓] XÁC MINH THÀNH CÔNG! 
Mã băm trong chữ ký khớp hoàn toàn với mã băm của nội dung chứng chỉ.
